# Introduction

## 什么是运动规划？

从根本上说，机械臂运动规划的核心问题是寻找一个有效且连续的配置序列（即路径或轨迹），该序列能够将机器人从一个起始状态移动到一个期望的目标状态，同时不违反任何约束条件，其中最关键的约束是避免与自身或环境中的障碍物发生碰撞 。这个基本问题可以进一步分解为两个子问题：路径规划（Path Planning）和轨迹规划（Trajectory Planning）。路径规划专注于寻找一条纯粹的几何路径，而轨迹规划则为这条路径赋予时间规律，即定义其速度和加速度曲线 。   

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 100%; height: auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/obs_avoid_example.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            Son D, Jung H, Kim B. NeuralSVCD for Efficient Swept Volume Collision Detection[J]. <br>
            arXiv preprint arXiv:2509.00499, 2025.
    </figcaption>
</figure>

在现代机器人技术中自主规划运动的能力，是将机器人从一个简单的遥控设备转变为智能体的关键。这项能力是现代机器人技术的基石，它催生了众多深刻影响生产力和社会的应用 。尤其是操作 (Manipulation) 方面。   

工业自动化：在制造业中，运动规划对于焊接、复杂零件组装和码垛等精密任务至关重要。在这些结构化环境中，机器人必须执行精确且可重复的运动 。 

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 100%; height: auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/tesla_factory.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            特斯拉上海工厂
    </figcaption>
</figure>

  

医疗健康：在医疗领域，机械臂辅助进行需要高度精确性的外科手术，帮助患者康复，并执行实验室自动化任务 。

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 100%; height: auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/davinci_robot.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            达芬奇手术机器人
    </figcaption>
</figure>

服务与协作机器人：随着机器人逐渐进入以人为中心的环境，挑战也随之加剧。协作机器人（Cobots）必须能够安全、可预测地在人类周围规划运动，并适应家庭、仓库和医院等动态、非结构化的环境 。   

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 100%; height: auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/figure3.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            Figure 03 服务机器人
    </figcaption>
</figure>


## 什么是“好的”规划？

运动规划研究的最初焦点仅仅是找到任何可行的解决方案，这一概念被称为“完备性”（Completeness）。比如最经典的 Dijkstra 算法

```{=html}
<iframe src="../web_demos/intro/dijkstra.html" title="Dijkstra 可视化"
        style="width: 95%; height: 1050px; border: 1px solid #e5e7eb; border-radius: 12px; box-shadow: 0 10px 25px rgba(0,0,0,0.08);
                display: block; margin: 0 auto; background: white;">
</iframe>
```

然而，现代应用的要求远不止于此。一个“好”的规划需要通过多个标准来评估：   

- 可行性（Feasibility）：规划出的路径必须是无碰撞的，并且要遵守机器人的运动学和动力学约束 。   
- 最优性（Optimality）：规划应在给定成本函数下达到“最优”。这可能意味着最小化路径长度、执行时间或能耗 。   
- 平滑性与可预测性（Smoothness & Predictability）：为了安全可靠地执行，轨迹应该是平滑的（即具有较低的加加速度和加速度）。在工业和协作场景中，可预测性至关重要；相似的规划查询应该产生相似的路径 。

<figure style="text-align: center; margin: 20px 0;">
    <img src="https://github.com/quimortiz/dynoplan/assets/32126190/b14905b7-8a8b-435e-be6e-11dfc49f909a" 
         width="80%" 
         style="display: block; margin: 0 auto;"> 
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
        iDb-A\* Optimal Trajectory Planning <br>
        de Haro J O, Hönig W, Hartmann V N, et al. <br>
        iDb-A*: Iterative search and optimization for optimal kinodynamic motion planning[J]. <br>IEEE Trans. Robotics, 2025.
    </figcaption>
</figure>


这种从可行性到多目标最优性的演变，清晰地反映了机器人技术领域的成熟过程。

早期的研究工作主要关注一个二元问题：“是否存在一条路径？”能够保证在路径存在时找到它的算法被称为“完备的”，但对于复杂机器人来说，这些算法的计算成本往往高得令人望而却步。这一计算瓶颈催生了基于采样的规划方法，它们牺牲了绝对的完备性，以换取概率完备性和实践中的高效率，但其初始解往往不平滑且非最优。

这些初始解的质量不佳，反过来又催生了对“更好”路径的需求，从而引入了最优性的概念（例如最短路径），并推动了像 RRT* 这样的算法的发展。

<figure style="text-align: center; margin: 20px 0;">
    <img src="https://daoming-chen.github.io/MP_lecture_assets/lec0/rrt_star.png" 
         width="80%" 
         style="display: block; margin: 0 auto;"> 
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
        RRT -> RRT* <br>
        Karaman S, Frazzoli E. Incremental sampling-based algorithms for optimal motion planning[J]. <br>
        Robotics Science and Systems VI, 2010, 104(2): 267-274.
    </figcaption>
</figure>


随着机器人变得更加动态，并以更复杂的方式与世界互动，单一的路径长度度量已不再足够。电机扭矩、能量消耗以及流畅运动所需的平滑度等因素变得至关重要。这最终导致了轨迹优化框架（如 CHOMP、STOMP、TrajOpt）的兴起，这些框架通过定义丰富的、包含多个项的成本函数来构建问题。

<figure style="text-align: center; margin: 20px 0;">
    <img src="https://daoming-chen.github.io/MP_lecture_assets/lec0/trajopt.gif" 
         width="80%" 
         style="display: block; margin: 0 auto;"> 
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
        TrajOpt <br>
        Schulman J, Duan Y, Ho J, et al. Motion planning with sequential convex optimization and convex collision checking[J]. <br>
        The International Journal of Robotics Research, 2014, 33(9): 1251-1270.
    </figcaption>
</figure>


这代表了现代运动规划的观点：它不再仅仅是一个搜索问题，而是一个整体的优化问题。这一趋势表明，随着机器人越来越多地在现实世界中承担复杂任务，运动的质量变得与运动的存在本身同等重要。这种转变要求规划器能够处理复杂的、通常是不可微的成本函数和约束。

## 什么是任务与运动规划（TAMP）？

前面的小节讨论了如何从 A 点到 B 点找到一条“好”的路径。但一个更根本的问题是：机器人 *为什么* 要去 A 点或 B 点？它在这些点要 *做什么*？

在经典的运动规划中，我们假设“任务”已经被给定（例如，“将物体从 X 移动到 Y”）。然而，在更复杂的场景中，机器人必须自己 *推理* 出需要执行哪些动作序列。例如，要“倒一杯水”，机器人必须首先找到杯子，然后移动到饮水机，按下按钮，最后再把水杯递过来。

**任务与运动规划（Task and Motion Planning, TAMP）** 正是连接这两个层面的桥梁：它在一个统一的框架内，同时解决高层次的、离散的“任务逻辑”（Task Planning，做什么）和低层次的、连续的“运动实现”（Motion Planning，怎么做）。

TAMP 的实现并非只有一种方式，其发展体现了机器人领域从“自动化”走向“智能化”的清晰脉络。

### 针对特定任务的“硬编码”

在传统的工业自动化中，TAMP 问题通常是被“规避”的。工程师会针对 特定 的任务类型（如抓取、码垛、焊接或打磨）编写高度专业化、逻辑僵硬的程序。这些程序本质上是一系列预先定义好的规则和状态机，它们在高度结构化、可预测的环境中运行良好。

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 100%; height: auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/3D%20Vision-Guided%20Bin%20Picking%20of%20Track%20Links%20with%20Mech-Mind.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            梅卡曼德 3D 相机无序抓取
    </figcaption>
</figure>

然而，这种方法的 *局限性* 非常明显。它依赖于工程师对 *已知* 任务的穷举编程。当机器人从工厂走向家庭、医院等非结构化环境时，任务的类型是无穷的。

### 运动原语与行为树（混合方法）

为了克服“硬编码”的僵硬性，同时又避免完全黑盒的复杂性，一种在工业界和研究中极为流行的混合方法应运而生：运动原语（Motion Primitives） 与 行为树（Behavior Trees） 的结合。
- 运动原语 (Motion Primitives)： 这不是去规划一个完整的、复杂轨迹，而是将复杂的动作分解为一系列“标准化的动作单元”。例如，“接近物体”、“线性移动”、“抓取”、“释放”等。这些原语本身是可以参数化的（比如“线性移动 10 厘米”），并且其内部的运动规划已经过充分优化和验证。
- 行为树 (Behavior Trees)： 行为树提供了一种强大的、模块化的方式来组织这些原语，以构建复杂的任务逻辑。它取代了传统有限状态机（FSM）难以维护和扩展的缺点。通过行为树，工程师可以用图形化的方式编排任务流程，定义动作的执行顺序、失败回退机制（Fallback）和并行操作。

这种方法（例如在 MoveIt Pro 等现代机器人框架中广泛应用）的优势在于它提供了 “乐高积木式” 的灵活性。工程师不再需要从头编写每一个焊接或码垛程序，而是可以像搭积木一样，将可靠的“运动原语”通过“行为树”快速组合起来，以适应不同的任务需求。

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 100%; height: auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/MoveIt%20Pro%20-%20Building%20a%20Door%20Opening%20Application%20with%20Behavior%20Trees.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            PicNic Moveit Pro
    </figcaption>
</figure>

尽管这种方法极大地提高了开发效率和系统的鲁棒性，但它仍然依赖于工程师对任务逻辑的 *显式设计*。

### 基于学习的 TAMP

当任务的复杂性呈指数级增长时——例如“叠被子”、“晾衣服”或“洗碗”——即便是行为树也难以手动编排。这些任务具有两个显著特点：
- 长时序（Long-horizon）： 它们需要一系列复杂的、有逻辑依赖的步骤才能完成。
- 高度不确定性（High Uncertainty）： 任务的上下文和环境是动态变化的（例如，被子在床上的形态、碗在水槽中的位置）。

这种从“编程逻辑”到“自主决策”的转变，是 TAMP 领域面临的核心挑战。如何让机器人自主 *推理* 出完成任务所需的动作序列（例如，要洗碗，必须 先拿起海绵，然后 挤上洗洁精），并为每一步自动生成可行且优化的运动轨迹？

这正是当前机器人研究的前沿，也是 *基于学习（Learning-based）*的方法 被寄予厚望的领域。研究者们希望通过模仿学习（Imitation Learning）或强化学习（Reinforcement Learning）等技术，让机器人不再依赖僵硬的编程规则，而是能从数据中“学会”如何规划和执行这些复杂的长时序任务。

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 100%; height: auto;" autoplay muted loop>
        <source src="https://website.pi-asset.com/real_time_chunking/sizzle.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            Physical Intelligence (π) <br>
            Real-Time Action Chunking with Large Models
    </figcaption>
</figure>

